In [2]:
# ==========================================================
# ACTIVIDAD 9. Clasificación definitiva y universo de acciones
# Combina el directorio de Nasdaq y la SEC. Asigna a cada
# candidato un motivo de inclusión o exclusión.
# ==========================================================
import re
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

# ----------------------------------------------------------
# 1. Unir las tres tablas de clasificación
# ----------------------------------------------------------
leer = lambda nombre: pd.read_csv(CARPETA / nombre, dtype=str, keep_default_na=False)
preliminar = leer("clasificacion_preliminar.csv")              # Actividad 6
nasdaq = leer("emparejamiento_nasdaq.csv")[["ticker", "simbolo_original"]]   # Actividad 5
sec = leer("clasificacion_sec.csv").drop(columns=["clase", "nombre_valor"])  # Actividad 8

datos = (preliminar.merge(nasdaq, on="ticker", how="left")
                   .merge(sec, on="ticker", how="left")
                   .fillna(""))
print("Candidatos:", len(datos), "| tickers únicos:", datos["ticker"].nunique())

# ----------------------------------------------------------
# 2. Reglas definitivas: se aplica la PRIMERA que coincida
# ----------------------------------------------------------
EXCLUIDAS_DIRECTORIO = ["sin_directorio", "etf", "prueba", "adr", "preferente", "fondo",
                        "mlp", "deuda", "unidad", "warrant", "derecho"]
SOBRESCRITURAS = {"TVE": "deuda"}   # Tennessee Valley Authority: lo que cotiza son bonos

def es_estado_eeuu(codigo):
    """Los estados de EE. UU. usan dos letras; los países extranjeros, letra y número."""
    return re.fullmatch(r"[A-Z]{2}", codigo) is not None

def motivo(fila):
    nombre = fila["nombre_valor"].lower()
    # a) Exclusiones del directorio de Nasdaq (Actividad 6)
    if fila["clase"] in EXCLUIDAS_DIRECTORIO:
        return "dir_" + fila["clase"]
    if fila["ticker"] in SOBRESCRITURAS:
        return SOBRESCRITURAS[fila["ticker"]]
    # b) Régimen de información en la SEC
    if fila["cik"] == "":
        return "sin_registro_sec"
    regimen = fila["regimen"]
    if regimen == "sin_informe_anual" and es_estado_eeuu(fila["constitucion"]) \
            and fila["tipo_entidad"] == "operating":
        regimen = "nacional"                       # nacional inferido
    if regimen != "nacional":
        return regimen                             # extranjero, fondo o sin_informe_anual
    # c) Emisores nacionales que no son acciones ordinarias
    if "$" in fila["simbolo_original"] or re.search(r"\bpfd\b|\bprd\b|preferred", nombre):
        return "preferente"
    if fila["sic"] == "6189" or re.search(r"capital trust|strats|corts|\btr ctf\b", nombre):
        return "estructurado"
    if fila["expediente_anual"].startswith("814-") or fila["sic"] == "":
        return "bdc"
    if fila["sic"] in ("6792", "6795") and "trust" in nombre:
        return "fideicomiso_regalias"
    if re.search(r"\bl\.\s?p\.|limited partner|common units", nombre):
        return "mlp"
    return "accion_ordinaria"

datos["motivo"] = datos.apply(motivo, axis=1)
datos["incluido"] = datos["motivo"] == "accion_ordinaria"

# ----------------------------------------------------------
# 3. Resultado de la clasificación
# ----------------------------------------------------------
print("\n=== MOTIVO FINAL ===")
print(datos["motivo"].value_counts())
print("\nIncluidos:", datos["incluido"].sum(), "de", len(datos))

inferidos = datos[(datos["regimen"] == "sin_informe_anual") & datos["incluido"]]
print("\nNacionales inferidos (sin informe anual, constituidos en EE. UU.):",
      sorted(inferidos["ticker"]))

# ----------------------------------------------------------
# 4. Universo: empresas, clases y sectores
# ----------------------------------------------------------
universo = datos[datos["incluido"]].copy()
universo["clases_emisor"] = universo.groupby("cik")["ticker"].transform("count")

def division_sic(sic):
    """División del sistema SIC según sus dos primeros dígitos."""
    if not sic.isdigit():
        return "Sin SIC"
    d = int(sic[:2])
    for limite, nombre in [(9, "A Agricultura"), (14, "B Minería"), (17, "C Construcción"),
                           (39, "D Manufactura"), (49, "E Transporte y servicios públicos"),
                           (51, "F Comercio mayorista"), (59, "G Comercio minorista"),
                           (67, "H Finanzas, seguros e inmobiliario"), (89, "I Servicios"),
                           (99, "J Administración pública")]:
        if d <= limite:
            return nombre

universo["division_sic"] = universo["sic"].apply(division_sic)

print("\n=== UNIVERSO DE ACCIONES ORDINARIAS ===")
print("Tickers:", len(universo), "| empresas (CIK):", universo["cik"].nunique())
varias = universo[universo["clases_emisor"] > 1].groupby("cik")["ticker"].agg(", ".join)
print("Empresas con varias clases:", len(varias), "| tickers implicados:",
      (universo["clases_emisor"] > 1).sum())
print(varias.head(10).to_string())

print("\nPor división SIC (tickers):")
print(universo["division_sic"].value_counts())
print("\nPor bolsa:")
print(universo["bolsa"].value_counts())

print("\n=== AUDITORÍA: 25 acciones incluidas al azar ===")
print(universo.sample(25, random_state=2026)[["ticker", "nombre_valor", "sic"]].to_string(index=False))

# ----------------------------------------------------------
# 5. Guardar
# ----------------------------------------------------------
columnas = ["ticker", "cik", "nombre_valor", "nombre_sec", "bolsa", "sic", "sector_sic",
            "constitucion", "regimen", "motivo", "incluido", "fecha_min", "fecha_max", "filas"]
datos[columnas].to_csv(CARPETA / "clasificacion_final.csv", index=False)
universo[columnas + ["clases_emisor", "division_sic"]].to_csv(CARPETA / "universo_acciones.csv", index=False)
print("\nGuardados: clasificacion_final.csv (todos los candidatos) y universo_acciones.csv (incluidos)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Candidatos: 4046 | tickers únicos: 4046

=== MOTIVO FINAL ===
motivo
accion_ordinaria        2823
extranjero               333
dir_fondo                237
dir_adr                  209
dir_preferente           179
fondo                     73
dir_sin_directorio        67
dir_mlp                   34
bdc                       20
dir_deuda                 18
estructurado               9
dir_unidad                 8
fideicomiso_regalias       7
preferente                 7
sin_registro_sec           6
sin_informe_anual          6
dir_etf                    5
mlp                        3
dir_derecho                1
deuda                      1
Name: count, dtype: int64

Incluidos: 2823 de 4046

Nacionales inferidos (sin informe anual, constituidos en EE. UU.): ['BSEM', 'CLBK', 'DMRC', 'NAVI', 'NVRI', 'PBAM', 'WCCB', 'XOM']

=== UNIVERSO DE ACCIONES ORDINARIAS ==